<a href="https://colab.research.google.com/github/sjschooley/SDS510/blob/main/Mod_5_basics_badge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The code for this assignment comes from 5.6, or from section 3 for the pandas work, unless otherwise noted.

In [1]:
import nltk
from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
import pandas as pd
import re
from string import punctuation
import json

In [2]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [3]:
with open('/content/jeopardy.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
df = pd.DataFrame(data)

In [6]:
df.head(25)

,category,air_date,question,value,answer,round,show_number
0,HISTORY,2004-12-31,"'For the last 8 years of his life, Galileo was...",$200,Copernicus,Jeopardy!,4680
1,ESPN's TOP 10 ALL-TIME ATHLETES,2004-12-31,'No. 2: 1912 Olympian; football star at Carlis...,$200,Jim Thorpe,Jeopardy!,4680
2,EVERYBODY TALKS ABOUT IT...,2004-12-31,'The city of Yuma in this state has a record a...,$200,Arizona,Jeopardy!,4680
3,THE COMPANY LINE,2004-12-31,"'In 1963, live on ""The Art Linkletter Show"", t...",$200,McDonald\'s,Jeopardy!,4680
4,EPITAPHS & TRIBUTES,2004-12-31,"'Signer of the Dec. of Indep., framer of the C...",$200,John Adams,Jeopardy!,4680
5,3-LETTER WORDS,2004-12-31,"'In the title of an Aesop fable, this insect s...",$200,the ant,Jeopardy!,4680
6,HISTORY,2004-12-31,'Built in 312 B.C. to link Rome & the South of...,$400,the Appian Way,Jeopardy!,4680
7,ESPN's TOP 10 ALL-TIME ATHLETES,2004-12-31,'No. 8: 30 steals for the Birmingham Barons; 2...,$400,Michael Jordan,Jeopardy!,4680
8,EVERYBODY TALKS ABOUT IT...,2004-12-31,"'In the winter of 1971-72, a record 1,122 inch...",$400,Washington,Jeopardy!,4680
9,THE COMPANY LINE,2004-12-31,'This housewares store was named for the packa...,$400,Crate & Barrel,Jeopardy!,4680


In [5]:
print(df.shape)

(216930, 7)


The assignment involves labelling questions as high value (intelligence), so the first question is what qualifies.  I thought about splitting the questions up by round, but then I realized that hte tiebreakers and final jeopardy questions have a value of None.  And some of the lower-value double jeopardy questions still have values under $1,000.  So maybe I was overthinking it.  I went with just calling questions with a monetary value over $1,000 high value.

In [10]:
lowval = [None, '$200', '$400', '$600', '$800', '$1000']

In [11]:
df['intel'] = [1 if x in lowval else 0 for x in df['value']]

In [12]:
df

,category,air_date,question,value,answer,round,show_number,intel
0,HISTORY,2004-12-31,"'For the last 8 years of his life, Galileo was...",$200,Copernicus,Jeopardy!,4680,1
1,ESPN's TOP 10 ALL-TIME ATHLETES,2004-12-31,'No. 2: 1912 Olympian; football star at Carlis...,$200,Jim Thorpe,Jeopardy!,4680,1
2,EVERYBODY TALKS ABOUT IT...,2004-12-31,'The city of Yuma in this state has a record a...,$200,Arizona,Jeopardy!,4680,1
3,THE COMPANY LINE,2004-12-31,"'In 1963, live on ""The Art Linkletter Show"", t...",$200,McDonald\'s,Jeopardy!,4680,1
4,EPITAPHS & TRIBUTES,2004-12-31,"'Signer of the Dec. of Indep., framer of the C...",$200,John Adams,Jeopardy!,4680,1
...,...,...,...,...,...,...,...,...
216925,RIDDLE ME THIS,2006-05-11,'This Puccini opera turns on the solution to 3...,$2000,Turandot,Double Jeopardy!,4999,0
216926,"""T"" BIRDS",2006-05-11,'In North America this term is properly applie...,$2000,a titmouse,Double Jeopardy!,4999,0
216927,AUTHORS IN THEIR YOUTH,2006-05-11,"'In Penny Lane, where this ""Hellraiser"" grew u...",$2000,Clive Barker,Double Jeopardy!,4999,0
216928,QUOTATIONS,2006-05-11,"'From Ft. Sill, Okla. he made the plea, Arizon...",$2000,Geronimo,Double Jeopardy!,4999,0


The first time I defined lowval, I put None in quotes, the same as the other values, but that didn't categorize correctly.  So I tried it without the quotes, because maybe None is an actual value, not a string, and that seems to have worked.

Now that I have my data, I need to clean it up.  I'll start by turning the question column from the dataframe into a list.  I see the quotes, so I know each question is already a string.

In [13]:
qlist = df['question'].tolist()
qlist

["'For the last 8 years of his life, Galileo was under house arrest for espousing this man's theory'",
 "'No. 2: 1912 Olympian; football star at Carlisle Indian School; 6 MLB seasons with the Reds, Giants & Braves'",
 "'The city of Yuma in this state has a record average of 4,055 hours of sunshine each year'",
 '\'In 1963, live on "The Art Linkletter Show", this company served its billionth burger\'',
 "'Signer of the Dec. of Indep., framer of the Constitution of Mass., second President of the United States'",
 "'In the title of an Aesop fable, this insect shared billing with a grasshopper'",
 "'Built in 312 B.C. to link Rome & the South of Italy, it's still in use today'",
 "'No. 8: 30 steals for the Birmingham Barons; 2,306 steals for the Bulls'",
 "'In the winter of 1971-72, a record 1,122 inches of snow fell at Rainier Paradise Ranger Station in this state'",
 "'This housewares store was named for the packaging its merchandise came in & was first displayed on'",
 '\'"And away we go

Looking at the text, there are some things that need to be added to the stopwords, or confirmed to be in the punctuation list.

In [14]:
english_stopwords = set(stopwords.words('english') + list(punctuation) + ['<br />'])

First step is to make eveything lowercase.

In [15]:
text.lower(qlist)

NameError: name 'text' is not defined

In [16]:
lower(qlist)

NameError: name 'lower' is not defined

So, at this point, I asked Gemini to explain the error.

The error AttributeError: 'list' object has no attribute 'text' occurs because qlist is a Python list of strings.

A standard Python list does not have an attribute called .text, nor does it have a .lower() method (which is a method of individual string objects, not lists).

If you want to convert every question inside qlist to lowercase, you must loop through the list and apply .lower() to each individual string. The most Pythonic way to do this is with a list comprehension.

From this I understood that lower is a method to use on a text, not a list.  So I have to use it on each text(string) in the list individually.

In [17]:
qlist_lower = [q.lower() for q in qlist]
# Display the first few to verify
print(qlist_lower[:3])

["'for the last 8 years of his life, galileo was under house arrest for espousing this man's theory'", "'no. 2: 1912 olympian; football star at carlisle indian school; 6 mlb seasons with the reds, giants & braves'", "'the city of yuma in this state has a record average of 4,055 hours of sunshine each year'"]


Next, word tokenize and lemmatize, pulling directly from the 5.6 code.

In [18]:
wordlist = [lemmatizer.lemmatize(x) for x in word_tokenize(qlist_lower)
      if x not in english_stopwords]

LookupError: 
**********************************************************************
  Resource [93mpunkt_tab[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('punkt_tab')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtokenizers/punkt_tab/english/[0m

  Searched in:
    - '/root/nltk_data'
    - '/usr/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************


Shockingly, I do understand one part of this error message, about missing resources.

In [19]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [20]:
wordlist = [lemmatizer.lemmatize(x) for x in word_tokenize(qlist_lower)
      if x not in english_stopwords]

TypeError: expected string or bytes-like object, got 'list'

So I think the problem is that the word tokenizer expected a string, but got a list instead.  But I thought this code would go through them individually.  So I'm not sure how to write that.

In [21]:
tokens = for each q in qlist_lower word_tokenize(q)
tokens

SyntaxError: invalid syntax (1604193959.py, line 1)

In [22]:
tokens = word_tokenize(q) for eachq in qlist_lower
tokens

SyntaxError: invalid syntax (1859788286.py, line 1)

I'm not sure what to do, so I asked Gemini for help with the original code I tried to use.

Response from Gemini: The error TypeError: expected string or bytes-like object, got 'list' is occurring because NLTK's word_tokenize() function expects a single string as its input, but you are passing qlist_lower (which is a list of strings).

To resolve this and tokenise/lemmatise your data, we should:

Initialize the WordNetLemmatizer (and download the necessary wordnet resource if it isn't already downloaded).
Process each question individually. We can define a clean, reusable helper function to tokenize, remove stopwords/punctuation, and lemmatize a single string.
Apply this helper function to your entire corpus using either a list comprehension or pandas .apply(), which is highly efficient for dataframes.

In [ ]:
wordlist = [lemmatizer.lemmatize(x) for x in word_tokenize(qlist_lower)
      if x not in english_stopwords]

TypeError: expected string or bytes-like object, got 'list'

In [ ]:
import nltk
# Download wordnet and omw-1.4 for the Lemmatizer
nltk.download('wordnet')
nltk.download('omw-1.4')

# Initialize the lemmatizer
lemmatizer = WordNetLemmatizer()

# Define a function to clean, tokenize, and lemmatize a single question text
def preprocess_text(text):
    # Tokenize the individual string
    tokens = word_tokenize(text)
    # Filter stopwords/punctuation and lemmatize remaining words
    cleaned_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in english_stopwords]
    return " ".join(cleaned_tokens)

# Process the lowered questions and store them in a new list
cleaned_qlist = [preprocess_text(q) for q in qlist_lower]

# Display the first 10 original and processed questions
for orig, clean in zip(qlist_lower[:10], cleaned_qlist[:10]):
    print(f"Original: {orig}")
    print(f"Cleaned:  {clean}\n")

So most of this I can follow, even I couldn't have written it, but I don't see why we have to define the lemmatizer.  I thought that was a method, and that it was already downloaded.

In [23]:
import nltk
# Download wordnet and omw-1.4 for the Lemmatizer
nltk.download('wordnet')
nltk.download('omw-1.4')

# Initialize the lemmatizer
lemmatizer = WordNetLemmatizer()

# Define a function to clean, tokenize, and lemmatize a single question text
def preprocess_text(text):
    # Tokenize the individual string
    tokens = word_tokenize(text)
    # Filter stopwords/punctuation and lemmatize remaining words
    cleaned_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in english_stopwords]
    return " ".join(cleaned_tokens)

# Process the lowered questions and store them in a new list
cleaned_qlist = [preprocess_text(q) for q in qlist_lower]

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [24]:
cleaned_qlist

["'for last 8 year life galileo house arrest espousing man 's theory",
 "'no 2 1912 olympian football star carlisle indian school 6 mlb season red giant brave",
 "'the city yuma state record average 4,055 hour sunshine year",
 "'in 1963 live `` art linkletter show '' company served billionth burger",
 "'signer dec. indep. framer constitution mass. second president united state",
 "'in title aesop fable insect shared billing grasshopper",
 "'built 312 b.c link rome south italy 's still use today",
 "'no 8 30 steal birmingham baron 2,306 steal bull",
 "'in winter 1971-72 record 1,122 inch snow fell rainier paradise ranger station state",
 "'this housewares store named packaging merchandise came first displayed",
 "'' away go ''",
 "'cows regurgitate first stomach mouth chew",
 "'in 1000 rajaraja cholas battled take indian ocean island known tea",
 "'no 1 lettered hoop football lacrosse syracuse think could n't act ask 11 `` unclean '' buddy",
 "'on june 28 1994 nat l weather service bega

A couple of problems here, so it looks like I need to re-clean.

In [31]:
english_stopwords = set(stopwords.words('english') + list(punctuation) + ['<br />','br','\'s','``','...','\'\''])

In [32]:
def preprocess_text(text):
    tokens = word_tokenize(text)
    cleaned_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in english_stopwords]
    return " ".join(cleaned_tokens)

cleaned_qlist = [preprocess_text(q) for q in qlist_lower]

In [33]:
cleaned_qlist

["'for last 8 year life galileo house arrest espousing man theory",
 "'no 2 1912 olympian football star carlisle indian school 6 mlb season red giant brave",
 "'the city yuma state record average 4,055 hour sunshine year",
 "'in 1963 live art linkletter show company served billionth burger",
 "'signer dec. indep. framer constitution mass. second president united state",
 "'in title aesop fable insect shared billing grasshopper",
 "'built 312 b.c link rome south italy still use today",
 "'no 8 30 steal birmingham baron 2,306 steal bull",
 "'in winter 1971-72 record 1,122 inch snow fell rainier paradise ranger station state",
 "'this housewares store named packaging merchandise came first displayed",
 'away go',
 "'cows regurgitate first stomach mouth chew",
 "'in 1000 rajaraja cholas battled take indian ocean island known tea",
 "'no 1 lettered hoop football lacrosse syracuse think could n't act ask 11 unclean buddy",
 "'on june 28 1994 nat l weather service began issuing index rate int

So I have a cleaner wordlist.  Now to set up and run Naive Bayes.

In [34]:
hvi = df['intel'].tolist()
hvi

[1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,


In [35]:
df2 = pd.DataFrame({'qintelval':hvi,'qtext':cleaned_qlist})

In [36]:
df2.head(10)

,qintelval,qtext
0,1,'for last 8 year life galileo house arrest esp...
1,1,'no 2 1912 olympian football star carlisle ind...
2,1,"'the city yuma state record average 4,055 hour..."
3,1,'in 1963 live art linkletter show company serv...
4,1,'signer dec. indep. framer constitution mass. ...
5,1,'in title aesop fable insect shared billing gr...
6,1,'built 312 b.c link rome south italy still use...
7,1,"'no 8 30 steal birmingham baron 2,306 steal bull"
8,1,"'in winter 1971-72 record 1,122 inch snow fell..."
9,1,'this housewares store named packaging merchan...


In [37]:
X_train, X_test, y_train, y_test = train_test_split(df2.qtext, df2.qintelval, random_state=1)

In [38]:
tfidf_vectorizer = TfidfVectorizer(use_idf=True)
X_train_tf = tfidf_vectorizer.fit_transform(X_train)
X_test_tf = tfidf_vectorizer.transform(X_test)

In [39]:
naive_bayes = MultinomialNB()
naive_bayes.fit(X_train_tf, y_train)
predictions = naive_bayes.predict(X_test_tf)

In [40]:
print('Accuracy: ', accuracy_score(y_test, predictions))

Accuracy:  0.6810429074548706
